# acc

In [3]:
import json

# 文件路径
qa_pairs_path = '/root/repo/MindSearch/Datasets/qa_pairs.json'
mindsearch_results_path = '/root/repo/MindSearch/Datasets/mindsearch_results_5.json'

# 读取 JSON 文件
with open(qa_pairs_path, 'r') as f:
    qa_pairs = json.load(f)

with open(mindsearch_results_path, 'r') as f:
    mindsearch_results = json.load(f)

# 创建一个字典来存储问题和答案对
qa_dict = {item['question']: item['answer'] for item in qa_pairs}

# 比较答案
results = []
correct_count = 0

for result in mindsearch_results:
    query = result.get('query', '')
    response = result.get('responses', '')
    actual_answer = response
    
    # # 提取实际回答
    # if isinstance(response, dict):
    #     actual_answer = response.get('response', {}).get('content', '')
    # else:
    #     actual_answer = response
    
    # 获取预期答案
    expected_answer = qa_dict.get(query, '')

    # 判断答案是否一致
    if expected_answer and actual_answer:
        # is_correct = expected_answer.lower() in str(actual_answer).lower()
        is_correct = expected_answer in actual_answer
    else:
        is_correct = False
    
    if is_correct:
        correct_count += 1
    
    results.append({
        'query': query,
        'expected_answer': expected_answer,
        'actual_answer': actual_answer,
        'is_correct': is_correct
    })

# 计算准确度
total_questions = len(mindsearch_results)
accuracy = correct_count / total_questions if total_questions > 0 else 0


# 筛选回答错误的问题
incorrect_results = [result for result in results if not result['is_correct']]

# 输出回答错误的问题
print("\nIncorrect Results:")
for result in incorrect_results:
    print(f"Query: {result['query']}")
    print(f"Expected Answer: {result['expected_answer']}")
    print(f"Actual Answer: {result['actual_answer']}")
    print(f"Is Correct: {result['is_correct']}")
    print('-' * 50)

# # 输出结果
# for result in results:
#     print(f"Query: {result['query']}")
#     print(f"Expected Answer: {result['expected_answer']}")
#     print(f"Actual Answer: {result['actual_answer']}")
#     print(f"Is Correct: {result['is_correct']}")
#     print('-' * 50)

print(f"Total Questions: {total_questions}")
print(f"Correct Answers: {correct_count}")
print(f"Incorrect Answers: {total_questions - correct_count}")
print(f"Accuracy: {accuracy:.2%}")


Incorrect Results:
Query: When did the last king from Britain's House of Hanover die?
Expected Answer: 20 June 1837
Actual Answer: The last king from Britain's House of Hanover was George V. He reigned as the King of Hanover from November 18, 1851, to September 20, 1866 [[1]][[2]]. King George V died on January 20, 1936. His death occurred at Sandringham House in Norfolk, England [[4]][[5]][[6]]. He had been suffering from chronic lung issues since 1928 and his health had gradually declined over the previous few months [[5]]. The exact cause of his death was a mixture of cocaine and morphine administered by his physician, Lord Bertrand Dawson, who intended to grant him a painless death [[7]][[8]].

Thus, the last king from Britain's House of Hanover died on January 20, 1936.
Is Correct: False
--------------------------------------------------
Query: How many people died in the second most powerful earthquake ever recorded?
Expected Answer: 131
Actual Answer: The second most powerful e

## 检测空答案
前期没有更改 prompt 以及 graph 的话，会因为无法分解单跳问题而导致回答为空，修改 prompt 当为单挑问题时，直接搜索 root 节点的原始问题即可。
还有 LLM 在生成代码的时候，经常不创建(add_node)就连接边(add_edge)和查看节点(nodes)，导致报错。
还有一种情况，查看节点(nodes())的时候， LLM 会出现幻觉，生成近义词节点名词，和之前创建和添加边的名词不一致，导致 Keyerror ，解决办法是在 graph 里面把从 nodes() 代码检测节点列表改成从 add_node() 代码检测节点列表，甚至我觉得 nodes 函数可以删除了，反正必须执行这个函数，那不如 add_node() 的时候就让 websearcher 搜索创建节点的 content 提问。

In [1]:
import json

# 加载数据
file_path = "/root/repo/MindSearch/Datasets/mindsearch_results.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 统计答案为空的数量
empty_responses_count = sum(1 for item in data if not item["responses"])
total_questions = len(data)

# 计算比例
empty_responses_ratio = empty_responses_count / total_questions if total_questions > 0 else 0

# 输出结果
print(f"总问题数: {total_questions}")
print(f"答案为空的问题数: {empty_responses_count}")
print(f"答案为空的问题比例: {empty_responses_ratio:.2%}")

总问题数: 125
答案为空的问题数: 9
答案为空的问题比例: 7.20%
